In [249]:
import pandas as pd
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from utils import get_all_jpg_pics,edge_extraction,detect_shapes
%matplotlib inline

In [250]:
IMGAE_ROOT = "data/Image/"
DATA_ROOT = "data/Scores/"

final_product_images = get_all_jpg_pics(IMGAE_ROOT+"Final-Product/")
df_final_product_scores = pd.read_csv(DATA_ROOT+"Final-product-scores.csv")

In [251]:
data_final_prod = []

for pic in final_product_images:
    img = cv2.imread(pic)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (0, 0), fx=0.2, fy=0.2)

    img, edge = edge_extraction(img, 10, 0)
    prosity, fibour = detect_shapes(edge)
    total = prosity + fibour
    porosity_score = prosity / total
    fiber_score = fibour / total
    mapped_score_fiber = 1 + (fiber_score * 9)
    mapped_score_porosity = 10 - (porosity_score * 9)

    data_final_prod.append({'Sample name': os.path.splitext(os.path.basename(pic))[0], 'Score_unsup_CV': mapped_score_fiber})
df_final_prod = pd.DataFrame(data_final_prod)

df_final_prod['Score'] = df_final_product_scores['Score']


In [ ]:
df_final_prod['TS'] = df_final_prod['Sample name'].str[:3]


In [253]:
timeseries_images = get_all_jpg_pics(IMGAE_ROOT+"Time-Based/")
df_timeseries_scores = pd.read_csv(DATA_ROOT+"Time-based-scores.csv")

In [254]:
data_timesseries = []

for pic in timeseries_images:
    img = cv2.imread(pic)

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (0, 0), fx=0.2, fy=0.2)

    img, edge = edge_extraction(img, 70, 100)
    prosity, fibour = detect_shapes(edge)
    total = prosity + fibour
    porosity_score = prosity / total
    fiber_score = fibour / total
    mapped_score_fiber = 1 + (fiber_score * 9)
    mapped_score_porosity = 10 - (porosity_score * 9)

    data_timesseries.append({'Sample name': os.path.splitext(os.path.basename(pic))[0], 'Score_unsup_CV': mapped_score_fiber})
df_timeseries = pd.DataFrame(data_timesseries)
df_timeseries['Score'] = df_timeseries_scores['Score']

In [ ]:
df_timeseries['TS'] = df_timeseries['Sample name'].str[:3]
df_unsup_cv_final = pd.concat([df_final_prod, df_timeseries], ignore_index=True)


In [256]:
df_unsup_cv_final['texture_unsup_CV']  = df_unsup_cv_final['Score_unsup_CV'].apply(lambda x: 0 if int(x) > 5 else 1)
df_unsup_cv_final['texture_real']  = df_unsup_cv_final['Score'].apply(lambda x: 0 if int(x) > 5 else 1)

In [257]:
df_unsup_cv_final.to_csv(DATA_ROOT+"Unsupervised_CV_scores.csv", index=False)